In [4]:
import pandas as pd
df_new = pd.read_csv("movie_data.csv", engine="python")

In [5]:
df_new.head()

,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre
0,Spider-Man: No Way Home,peter parker is unmasked and no longer able to...,5083.954,8940,8.3,en,action
1,The Batman,"in his second year of fighting crime, batman u...",3827.658,1151,8.1,en,crime
2,No Exit,stranded at a rest stop in the mountains durin...,2618.087,122,6.3,en,thriller
3,Encanto,"the tale of an extraordinary family, the madri...",2402.201,5076,7.7,en,animation
4,The King's Man,as a collection of history's worst tyrants and...,1895.511,1793,7.0,en,action


In [6]:
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

from scipy.sparse import csr_matrix, hstack

In [7]:
genre = df_new["Genre"].str.get_dummies(sep=" ")
genre = csr_matrix(genre.values)

language = pd.get_dummies(df_new["Original_Language"], prefix="Language", dtype=int)
language = csr_matrix(language.values)

tfidf = TfidfVectorizer(stop_words="english")
overview = tfidf.fit_transform(df_new["Overview"])

scaler = StandardScaler()
numeric = scaler.fit_transform(
    df_new[["Popularity","Vote_Count","Vote_Average"]]
)
numeric = csr_matrix(numeric)

from scipy.sparse import hstack
X = hstack([genre, language, overview, numeric])

dbscan = DBSCAN(eps=1.5, min_samples=5)
clusters = dbscan.fit_predict(X)

df_new["Cluster"] = clusters

In [8]:
print("clusters",len(set(dbscan.labels_)))

clusters 92


In [9]:
df_new["Cluster"].value_counts()

Cluster
 9     1225
-1     1183
 11    1168
 6     1070
 0      721
       ... 
 82       5
 83       5
 89       5
 90       5
 1        3
Name: count, Length: 92, dtype: int64

In [10]:
import random

def recommend_movie(movie_name):

    # User input ko lowercase karo
    movie_name = movie_name.lower()

    # Search movie
    movie = df_new[
        df_new["Title"].str.lower().str.contains(movie_name, na=False)
    ]

    if movie.empty:
        print("❌ Movie not found!")
        return

    # Movie ka cluster nikaalo
    cluster = movie["Cluster"].values[0]

    # Usi cluster ki movies
    cluster_movies = df_new[df_new["Cluster"] == cluster]

    # Wohi movie hata do
    cluster_movies = cluster_movies[
        cluster_movies["Title"].str.lower() != movie_name
    ]

    # Agar 5 se zyada movies hain
    if len(cluster_movies) >= 5:
        recommendations = random.sample(
            list(cluster_movies["Title"]),
            5
        )
    else:
        recommendations = list(cluster_movies["Title"])

    print("\n🎬 Recommended Movies\n")

    for i, movie in enumerate(recommendations, start=1):
        print(f"{i}. {movie}")

In [11]:
recommend_movie("batman")


🎬 Recommended Movies

1. Recalled
2. Spider-Man
3. Fifty Shades of Black
4. The Three Deaths of Marisela Escobedo
5. Headhunters


In [12]:
df_new.to_csv("movies_clusters.csv", index=False)